Environment Verification and Core Imports

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-Learn version: {sklearn.__version__}")

Load Data and Define Schema Constants

In [ ]:
# File path definition
DATA_PATH = "../data/customer_360_ml_workshop.csv"
df = pd.read_csv(DATA_PATH)

# Identity and target column definitions
ID_COL = "CustomerID"
TARGET_CHURN = "Churn"
TARGET_REVENUE = "Future12MRevenueEGP"

# Numerical features requiring median imputation and standard scaling
NUMERIC_FEATURES = [
    "Age",
    "TenureMonths",
    "MonthlyUsageGB",
    "MonthlyChargeEGP",
    "NumServices",
    "SupportCalls6M",
    "LatePayments12M",
    "SatisfactionScore",
]

# Categorical features requiring mode imputation and one-hot encoding
# Note: PaperlessBilling is explicitly incorporated based on Part A audit
CATEGORICAL_FEATURES = [
    "Region",
    "ContractType",
    "InternetType",
    "PaperlessBilling",
    "AutoPay",
]

print(f"Input records: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Numerical features count: {len(NUMERIC_FEATURES)}")
print(f"Categorical features count: {len(CATEGORICAL_FEATURES)}")

Data Integrity and Schema Audit

In [ ]:
# Ensure all defined modeling features exist in the dataframe
required_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing_cols = [c for c in required_columns if c not in df.columns]

if missing_cols:
    raise ValueError(f"Schema verification failed. Missing columns: {missing_cols}")

# Verify known missing value distributions identified in Part A
missing_summary = df[required_columns].isnull().sum()
print("Features containing missing values:")
print(missing_summary[missing_summary > 0])

Train/Test Split (Preventing Target Leakage)

In [ ]:
# Extract modeling features matrix X and classification target y
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = (df[TARGET_CHURN] == "Yes").astype(int)

# Stratified split to mirror the 18.13% churn distribution in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train dimensions: {X_train.shape}")
print(f"X_test dimensions: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.4f}")
print(f"Test churn rate: {y_test.mean():.4f}")

Build Reusable Preprocessing Pipeline

In [ ]:
# Numerical pipeline: Median imputation followed by standard scaling
num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical pipeline: Most frequent imputation followed by OneHotEncoding
# Compatible with scikit-learn 1.0.2: sparse=False avoids deprecation issues
cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ]
)

# Construct composite preprocessor via ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, NUMERIC_FEATURES),
        ("cat", cat_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

Fit and Transform Execution

In [ ]:
# Strictly fit on training set only, then transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test set using statistics calculated from training set
X_test_processed = preprocessor.transform(X_test)

print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"Processed X_test shape: {X_test_processed.shape}")
print(f"Remaining nulls in training array: {np.isnan(X_train_processed).sum()}")
print(f"Remaining nulls in testing array: {np.isnan(X_test_processed).sum()}")

Feature Name Reconstruction

In [ ]:
def extract_feature_names(fitted_transformer):
    """
    Reconstruct transformed feature column names without relying on
    newer get_feature_names_out() methods incompatible with sklearn 1.0.2.
    """
    feature_names = []

    # 1. Add numeric feature names
    feature_names.extend(NUMERIC_FEATURES)

    # 2. Add one-hot encoded categorical feature names
    cat_encoder = fitted_transformer.named_transformers_["cat"].named_steps["encoder"]
    for col_name, categories in zip(CATEGORICAL_FEATURES, cat_encoder.categories_):
        for category in categories:
            feature_names.append(f"{col_name}_{category}")

    return feature_names

feature_names = extract_feature_names(preprocessor)
print(f"Total engineered features: {len(feature_names)}")
print("Engineered feature schema:", feature_names)

DataFrame Construction

In [ ]:
# Convert processed training matrix to DataFrame for downstream verification
df_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

# Display sample of preprocessed data
df_train_processed.head()